<a href="https://colab.research.google.com/github/DVORA-AZARKOVICH/Narrative-Similarity/blob/main/Narrative_Similarity_Ensamble.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Narrative Similarity: Ensemble Strategy
**SemEval-2026 Task 4: Narrative Similarity**

This notebook implements an **Ensemble** approach to maximize accuracy.
We combine the predictions from our different pipelines (e.g., "Distill-then-Embed" and "Direct Prompting").

**The Logic:**
Different models make different types of errors. By combining them, we can "smooth out" the noise.
**Methods:**
1.  **Soft Voting:** Averaging the confidence scores (or probabilities) from multiple models.
2.  **Hard Voting:** Taking the majority vote (0 or 1) from the models.[link text](https://)

## 1. Setup and Load Predictions
We import pandas and load the result files generated by the previous notebooks.
We ensure all dataframes are sorted by `id` so we can merge them correctly.

In [ ]:
!pip install -q -U google-generativeai openai anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.1/155.1 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 51.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 388.2/388.2 kB 26.6 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import json
import requests
import time
import re
import random
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
from google.colab import userdata, drive
from openai import OpenAI
from anthropic import Anthropic

In [ ]:
try:
    drive.mount('/content/drive')
except:
    pass

BASE_PATH = '/content/drive/MyDrive/Narrative Similarity Data/'
DEV_FILE = BASE_PATH + 'SemEval2026-Task_4-dev-v1/dev_track_a.jsonl'
TEST_FILE = BASE_PATH + 'SemEval2026-Task_4-test-v1/test_track_a.jsonl'

df_dev = pd.read_json(DEV_FILE, lines=True)
df_test = pd.read_json(TEST_FILE, lines=True)
print(f"Loaded DataFrame shape: {df_dev.shape}")
print(f"Loaded DataFrame shape: {df_test.shape}")

Mounted at /content/drive
Loaded DataFrame shape: (200, 4)
Loaded DataFrame shape: (400, 3)


In [ ]:
try:
    GOOGLE_KEY = userdata.get('GOOGLE_API_KEY')
    OPENAI_KEY = userdata.get('OPENAI_API_KEY')
    ANTHROPIC_KEY = userdata.get('ANTHROPIC_API_KEY')
except userdata.SecretNotFoundError:
    print("Warning: Some keys are missing in Colab Secrets.")
    GOOGLE_KEY = None
    OPENAI_KEY = None
    ANTHROPIC_KEY = None

openai_client = OpenAI(api_key=OPENAI_KEY) if OPENAI_KEY else None
anthropic_client = Anthropic(api_key=ANTHROPIC_KEY) if ANTHROPIC_KEY else None

print("Clients initialized.")

Clients initialized.


## 2. Correlation Analysis
Before ensembling, we check how much the models agree with each other.
* **High Agreement:** Models are confident (or making the same mistakes).
* **Low Agreement:** The ensemble has a good chance of improving results by picking the "surer" model in ambiguous cases.

In [ ]:
MODEL_GEMINI = "gemini-2.5-flash"
MODEL_GPT = "gpt-4o"
MODEL_CLAUDE = "claude-sonnet-4-5-20250929"

SYS_PROMPT_TEXT = """
You are an expert annotator tasked with identifying narrative similarity between stories.
Your goal is to determine which of two candidate stories (Text A or Text B) is narratively closer to an Anchor story.

### DEFINITIONS OF NARRATIVE SIMILARITY
You must evaluate similarity based ONLY on the following three core aspects:
1. **Abstract Theme**: The defining constellation of problems, central ideas, and core motifs.
2. **Course of Action**: The sequence of events, actions, conflicts, turning points.
3. **Outcomes**: The results of the plot at the end of the text.

### INSTRUCTIONS
1. Analyze the Anchor.
2. Compare Text A to Anchor.
3. Compare Text B to Anchor.
4. Decide which text is narratively closer overall.

### INPUT DATA
**Anchor Text:**
{{INSERT_ANCHOR_TEXT_HERE}}

**Text A:**
{{INSERT_TEXT_A_HERE}}

**Text B:**
{{INSERT_TEXT_B_HERE}}

### OUTPUT FORMAT
Provide your response in valid JSON format:
{
  "reasoning": "Brief explanation.",
  "text_a_is_closer": boolean
}
"""

def clean_json_text(text):
    """מנקה סימני Markdown מתוך תשובת JSON"""
    text = text.strip()
    if text.startswith("```"):
        text = re.sub(r"^```(json)?", "", text)
        text = re.sub(r"```$", "", text)
    return text.strip()

In [ ]:

def get_gemini_prediction(anchor, text_a, text_b, row_index):
    if not GOOGLE_KEY:
        tqdm.write(f"⚠️ [Row {row_index}] Gemini: Missing Key")
        return None

    url = f"https://generativelanguage.googleapis.com/v1beta/models/{MODEL_GEMINI}:generateContent?key={GOOGLE_KEY}"
    headers = {"Content-Type": "application/json"}
    prompt_filled = f"Anchor:\n{anchor}\n\nText A:\n{text_a}\n\nText B:\n{text_b}"

    payload = {
        "system_instruction": {"parts": [{"text": SYS_PROMPT_TEXT}]},
        "contents": [{"parts": [{"text": prompt_filled}]}],
        "generationConfig": {"response_mime_type": "application/json"}
    }

    tqdm.write(f"🚀 [Row {row_index}] Gemini: Sending request...")

    for attempt in range(3):
        try:
            response = requests.post(url, headers=headers, json=payload, timeout=30)
            if response.status_code == 200:
                res_json = response.json()
                content = res_json['candidates'][0]['content']['parts'][0]['text']

                tqdm.write(f"✅ [Row {row_index}] Gemini Received: {content[:100]}...") # מציג את תחילת התשובה

                parsed = json.loads(clean_json_text(content))
                return parsed.get("text_a_is_closer")
            elif response.status_code in [429, 503]:
                tqdm.write(f"⏳ [Row {row_index}] Gemini: Rate limit (Attempt {attempt+1})")
                time.sleep(2 * (attempt + 1))
                continue
            else:
                tqdm.write(f"❌ [Row {row_index}] Gemini Error: {response.status_code} - {response.text}")
                return None
        except Exception as e:
            tqdm.write(f"❌ [Row {row_index}] Gemini Exception: {str(e)}")
            time.sleep(1)
    return None

def get_gpt_prediction(anchor, text_a, text_b, row_index):
    if not openai_client: return None
    prompt_filled = f"Anchor:\n{anchor}\n\nText A:\n{text_a}\n\nText B:\n{text_b}"

    tqdm.write(f"🚀 [Row {row_index}] GPT: Sending request...")

    try:
        completion = openai_client.chat.completions.create(
            model=MODEL_GPT,
            messages=[
                {"role": "system", "content": SYS_PROMPT_TEXT},
                {"role": "user", "content": prompt_filled}
            ],
            response_format={"type": "json_object"},
            temperature=0.0
        )
        content = completion.choices[0].message.content
        tqdm.write(f"✅ [Row {row_index}] GPT Received: {content[:100]}...")

        parsed = json.loads(content)
        return parsed.get("text_a_is_closer")
    except Exception as e:
        tqdm.write(f"❌ [Row {row_index}] GPT Error: {str(e)}")
        return None

def get_claude_prediction(anchor, text_a, text_b, row_index):
    if not anthropic_client: return None
    prompt_filled = f"Anchor:\n{anchor}\n\nText A:\n{text_a}\n\nText B:\n{text_b}"
    sys_clean = SYS_PROMPT_TEXT.replace("{{INSERT_ANCHOR_TEXT_HERE}}", "").replace("{{INSERT_TEXT_A_HERE}}", "").replace("{{INSERT_TEXT_B_HERE}}", "")

    tqdm.write(f"🚀 [Row {row_index}] Claude: Sending request...")

    try:
        message = anthropic_client.messages.create(
            model=MODEL_CLAUDE,
            max_tokens=1024,
            system=sys_clean,
            messages=[{"role": "user", "content": prompt_filled}]
        )
        content = message.content[0].text
        tqdm.write(f"✅ [Row {row_index}] Claude Received: {content[:100]}...")

        start = content.find('{')
        end = content.rfind('}') + 1
        if start != -1 and end != -1:
            return json.loads(content[start:end]).get("text_a_is_closer")
    except Exception as e:
        tqdm.write(f"❌ [Row {row_index}] Claude Error: {str(e)}")
        return None

In [ ]:

def get_ensemble_prediction(row, index):
    anchor = row['anchor_text']
    text_a = row['text_a']
    text_b = row['text_b']

    v_gemini = get_gemini_prediction(anchor, text_a, text_b, index)
    v_gpt = get_gpt_prediction(anchor, text_a, text_b, index)
    v_claude = get_claude_prediction(anchor, text_a, text_b, index)

    votes = [v for v in [v_gemini, v_gpt, v_claude] if v is not None]

    details = {"gemini": v_gemini, "gpt": v_gpt, "claude": v_claude}

    tqdm.write(f"🏁 [Row {index}] Votes: {details}")

    if not votes:
        return index, False, {**details, "FAILED_ALL": True}

    true_count = sum(1 for v in votes if v is True)
    required = len(votes) / 2
    final_pred = true_count > required

    if len(votes) == 2 and true_count == 1:
        if v_claude is not None: final_pred = v_claude
        elif v_gpt is not None: final_pred = v_gpt
        else: final_pred = v_gemini

    return index, final_pred, details

## 3. Final Evaluation
We evaluate the ensemble's performance against the ground truth.
Usually, this yields a higher F1-score than any single model alone.

In [ ]:
print("--- Starting Robust Ensemble Prediction ---")
rows = df_dev.to_dict('records')
results_map = {}
details_map = {}

with ThreadPoolExecutor(max_workers=4) as executor:
    futures = [executor.submit(get_ensemble_prediction, row, i) for i, row in enumerate(rows)]
    for future in tqdm(as_completed(futures), total=len(rows)):
        idx, res, det = future.result()
        results_map[idx] = res
        details_map[idx] = det

df_dev['prediction'] = [results_map[i] for i in range(len(rows))]
df_dev['votes_detail'] = [str(details_map[i]) for i in range(len(rows))]

print("\n--- Done ---")

--- Starting Robust Ensemble Prediction ---
🚀 [Row 0] Gemini: Sending request...
🚀 [Row 1] Gemini: Sending request...
🚀 [Row 2] Gemini: Sending request...
🚀 [Row 3] Gemini: Sending request...


  0%|          | 0/200 [00:00<?, ?it/s]

✅ [Row 63] Claude Received: ```json
{
  "reasoning": "The Anchor centers on an extramarital affair between two damaged individua...
🏁 [Row 63] Votes: {'gemini': True, 'gpt': True, 'claude': True}
🚀 [Row 67] Gemini: Sending request...


  0%|          | 0/200 [00:01<?, ?it/s]

⏳ [Row 67] Gemini: Rate limit (Attempt 1)


...

✅ [Row 198] GPT Received: {
  "reasoning": "The Anchor text focuses on the life and work of Tirumalai Krishnamacharya, explori...
🚀 [Row 198] Claude: Sending request...
✅ [Row 199] GPT Received: {
  "reasoning": "The Anchor story revolves around a political campaign with themes of scandal, blac...
🚀 [Row 199] Claude: Sending request...


 99%|█████████▉| 198/200 [24:13<00:16,  8.26s/it]

✅ [Row 197] Claude Received: ```json
{
  "reasoning": "The Anchor presents a thematic allegory about the stages of human life, wh...
🏁 [Row 197] Votes: {'gemini': True, 'gpt': False, 'claude': True}


100%|██████████| 200/200 [24:15<00:00,  7.28s/it]

✅ [Row 198] Claude Received: ```json
{
  "reasoning": "The Anchor text focuses on a documentary exploring the life and work of yo...
🏁 [Row 198] Votes: {'gemini': False, 'gpt': False, 'claude': False}
✅ [Row 199] Claude Received: ```json
{
  "reasoning": "The Anchor centers on a politician facing blackmail from a former mentor w...
🏁 [Row 199] Votes: {'gemini': False, 'gpt': True, 'claude': False}

--- Done ---


## 3. Final Evaluation
We evaluate the ensemble's performance against the ground truth.
Usually, this yields a higher F1-score than any single model alone.

In [ ]:
if 'text_a_is_closer' in df_dev.columns:
    acc = (df_dev['text_a_is_closer'] == df_dev['prediction']).mean()
    print(f"Final Accuracy: {acc:.4f}")

failed_count = sum(1 for d in details_map.values() if "FAILED_ALL" in d)
print(f"Rows with complete API failure (fallback used): {failed_count}")

import zipfile
from google.colab import files

sub_df = df_dev[['prediction']].rename(columns={'prediction': 'text_a_is_closer'})
sub_df.to_json('track_a.jsonl', orient='records', lines=True)

with zipfile.ZipFile('submission_robust.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    z.write('track_a.jsonl')

print("Created submission_robust.zip")
files.download('submission_robust.zip')

Final Accuracy: 0.7700
Rows with complete API failure (fallback used): 0
Created submission_robust.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import ast

def parse_vote_details(detail_str):
    try:
        return ast.literal_eval(detail_str)
    except:
        return {}

logs_df = df_dev.copy()

votes_expanded = logs_df['votes_detail'].apply(parse_vote_details).apply(pd.Series)

detailed_logs = pd.concat([
    logs_df[['anchor_text', 'text_a_is_closer', 'prediction']],
    votes_expanded
], axis=1)

detailed_logs = detailed_logs.rename(columns={
    'gemini': 'Vote_Gemini',
    'gpt': 'Vote_GPT',
    'claude': 'Vote_Claude',
    'prediction': 'Final_Ensemble_Prediction',
    'text_a_is_closer': 'Ground_Truth'
})

def has_disagreement(row):
    votes = [row['Vote_Gemini'], row['Vote_GPT'], row['Vote_Claude']]
    valid_votes = [v for v in votes if v is not None]
    if not valid_votes: return False
    return len(set(valid_votes)) > 1

disagreements = detailed_logs[detailed_logs.apply(has_disagreement, axis=1)]

print(f"Total rows with disagreement: {len(disagreements)}")
if not disagreements.empty:
    print("\n--- Sample Disagreements ---")
    display(disagreements[['Vote_Gemini', 'Vote_GPT', 'Vote_Claude', 'Final_Ensemble_Prediction', 'Ground_Truth']].head())

log_filename = 'full_model_logs.csv'
detailed_logs.to_csv(log_filename, index=True)

print(f"\nFull logs saved to {log_filename}")
files.download(log_filename)

Total rows with disagreement: 75

--- Sample Disagreements ---


,Vote_Gemini,Vote_GPT,Vote_Claude,Final_Ensemble_Prediction,Ground_Truth
1,True,True,False,True,True
3,True,False,False,False,False
5,True,False,True,True,False
9,None,True,False,False,True
11,None,False,True,True,True



Full logs saved to full_model_logs.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
print("--- Starting Robust Ensemble Prediction ---")
rows = df_test.to_dict('records')
results_map = {}
details_map = {}

with ThreadPoolExecutor(max_workers=4) as executor:
    futures = [executor.submit(get_ensemble_prediction, row, i) for i, row in enumerate(rows)]
    for future in tqdm(as_completed(futures), total=len(rows)):
        idx, res, det = future.result()
        results_map[idx] = res
        details_map[idx] = det

df_test['prediction'] = [results_map[i] for i in range(len(rows))]
df_test['votes_detail'] = [str(details_map[i]) for i in range(len(rows))]

print("\n--- Done ---")

--- Starting Robust Ensemble Prediction ---
🚀 [Row 0] Gemini: Sending request...
🚀 [Row 1] Gemini: Sending request...
🚀 [Row 2] Gemini: Sending request...
🚀 [Row 3] Gemini: Sending request...


  0%|          | 0/400 [00:08<?, ?it/s]

✅ [Row 0] Gemini Received: {
  "reasoning": "Text B shares significant narrative elements with the Anchor. Both are set in a hi...
🚀 [Row 0] GPT: Sending request...


  0%|          | 0/400 [00:09<?, ?it/s]

✅ [Row 2] Gemini Received: {
  "reasoning": "Text B is narratively closer to the Anchor. Both the Anchor and Text B feature a r...
🚀 [Row 2] GPT: Sending request...


...

✅ [Row 398] Claude Received: ```json
{
  "reasoning": "The Anchor focuses on two traumatized individuals (Mattia and Alice) who m...
🏁 [Row 398] Votes: {'gemini': False, 'gpt': False, 'claude': False}


100%|█████████▉| 399/400 [50:59<00:06,  6.58s/it]

✅ [Row 397] Claude Received: ```json
{
  "reasoning": "The Anchor centers on a comedic outing derailed by repeated automobile tro...
🏁 [Row 397] Votes: {'gemini': True, 'gpt': False, 'claude': False}


100%|██████████| 400/400 [51:03<00:00,  7.66s/it]

✅ [Row 399] Claude Received: ```json
{
  "reasoning": "The Anchor centers on a mysterious stranger who methodically targets multi...
🏁 [Row 399] Votes: {'gemini': True, 'gpt': True, 'claude': True}

--- Done ---


In [ ]:
# חישוב דיוק
if 'text_a_is_closer' in df_test.columns:
    acc = (df_test['text_a_is_closer'] == df_test['prediction']).mean()
    print(f"Final Accuracy: {acc:.4f}")

# בדיקה כמה נכשלו לחלוטין
failed_count = sum(1 for d in details_map.values() if "FAILED_ALL" in d)
print(f"Rows with complete API failure (fallback used): {failed_count}")

# יצירת קובץ להגשה
import zipfile
from google.colab import files

sub_df = df_test[['prediction']].rename(columns={'prediction': 'text_a_is_closer'})
sub_df.to_json('track_a.jsonl', orient='records', lines=True)

with zipfile.ZipFile('submission_robust_test.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    z.write('track_a.jsonl')

print("Created submission_robust test.zip")
files.download('submission_robust_test.zip')

Rows with complete API failure (fallback used): 0
Created submission_robust test.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import ast

def parse_vote_details(detail_str):
    try:
        return ast.literal_eval(detail_str)
    except:
        return {}

logs_df = df_test.copy()

votes_expanded = logs_df['votes_detail'].apply(parse_vote_details).apply(pd.Series)

detailed_logs = pd.concat([
    logs_df[['anchor_text', 'prediction']],
    votes_expanded
], axis=1)

detailed_logs = detailed_logs.rename(columns={
    'gemini': 'Vote_Gemini',
    'gpt': 'Vote_GPT',
    'claude': 'Vote_Claude',
    'prediction': 'Final_Ensemble_Prediction',
})

def has_disagreement(row):
    votes = [row['Vote_Gemini'], row['Vote_GPT'], row['Vote_Claude']]
    valid_votes = [v for v in votes if v is not None]
    if not valid_votes: return False
    return len(set(valid_votes)) > 1

disagreements = detailed_logs[detailed_logs.apply(has_disagreement, axis=1)]

print(f"Total rows with disagreement: {len(disagreements)}")
if not disagreements.empty:
    print("\n--- Sample Disagreements ---")
    display(disagreements[['Vote_Gemini', 'Vote_GPT', 'Vote_Claude', 'Final_Ensemble_Prediction']].head())

# 4. שמירה לקובץ CSV והורדה
log_filename = 'full_model_logs.csv'
detailed_logs.to_csv(log_filename, index=True)

print(f"\nFull logs saved to {log_filename}")
files.download(log_filename)

Total rows with disagreement: 144

--- Sample Disagreements ---


,Vote_Gemini,Vote_GPT,Vote_Claude,Final_Ensemble_Prediction
3,False,False,True,False
5,True,False,True,True
6,False,True,True,True
7,True,False,False,False
11,True,False,False,False



Full logs saved to full_model_logs.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>